### SVM

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [8]:
# Load datasets
austin_df = pd.read_csv("/Users/sohanaziz/Documents/AI for leaders sem 2/Datasets/cleaned datasets/week 2/cleaned_austin_data.csv")
airbnb_df = pd.read_csv("/Users/sohanaziz/Documents/AI for leaders sem 2/Datasets/cleaned datasets/week 2/listings_cleaned.csv")

# Define binary target creation functions
def create_binary_target(df, price_col):
    median_price = df[price_col].median()
    df['high_price'] = (df[price_col] > median_price).astype(int)
    return df

# Apply to datasets
austin_df = create_binary_target(austin_df, 'latestPrice')
airbnb_df = create_binary_target(airbnb_df, 'price')

In [5]:
# ----------- SVM Classification Preprocessing -----------

def preprocess_for_svm(df, target):
    df = df.dropna()
    df_numeric = df.select_dtypes(include=[np.number])
    X = df_numeric.drop(columns=[target])
    y = df_numeric[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, y_train, y_test


# Austin classification data
X_train_austin, X_test_austin, y_train_austin, y_test_austin = preprocess_for_svm(austin_df, 'high_price')

# Airbnb classification data (drop objects + id before numeric selection)
airbnb_drop_cols = airbnb_df.select_dtypes(include=['object']).columns.tolist() + ['id']
airbnb_classification_df = airbnb_df.drop(columns=airbnb_drop_cols, errors='ignore')
X_train_airbnb, X_test_airbnb, y_train_airbnb, y_test_airbnb = preprocess_for_svm(airbnb_classification_df, 'high_price')


# ----------- SVM Classification Modeling -----------
svm_classifier = SVC(kernel='rbf', C=1.0, gamma='scale')

# Austin SVM classification
svm_classifier.fit(X_train_austin, y_train_austin)
y_pred_austin = svm_classifier.predict(X_test_austin)
print("----- Austin SVM Classification Results -----")
print("Accuracy:", accuracy_score(y_test_austin, y_pred_austin))
print("Confusion Matrix:\n", confusion_matrix(y_test_austin, y_pred_austin))
print("Classification Report:\n", classification_report(y_test_austin, y_pred_austin))

# Airbnb SVM classification
svm_classifier.fit(X_train_airbnb, y_train_airbnb)
y_pred_airbnb = svm_classifier.predict(X_test_airbnb)
print("\n----- Airbnb SVM Classification Results -----")
print("Accuracy:", accuracy_score(y_test_airbnb, y_pred_airbnb))
print("Confusion Matrix:\n", confusion_matrix(y_test_airbnb, y_pred_airbnb))
print("Classification Report:\n", classification_report(y_test_airbnb, y_pred_airbnb))


----- Austin SVM Classification Results -----
Accuracy: 0.9413509060955519
Confusion Matrix:
 [[1457   77]
 [ 101 1400]]
Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.95      0.94      1534
           1       0.95      0.93      0.94      1501

    accuracy                           0.94      3035
   macro avg       0.94      0.94      0.94      3035
weighted avg       0.94      0.94      0.94      3035


----- Airbnb SVM Classification Results -----
Accuracy: 0.8525773195876288
Confusion Matrix:
 [[873 122]
 [164 781]]
Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.88      0.86       995
           1       0.86      0.83      0.85       945

    accuracy                           0.85      1940
   macro avg       0.85      0.85      0.85      1940
weighted avg       0.85      0.85      0.85      1940



In [6]:
# ----------- SVR Regression Preprocessing -----------

def preprocess_for_svr(df, target, drop_cols=[]):
    df = df.drop(columns=drop_cols, errors='ignore').dropna()
    X = df.select_dtypes(include=[np.number]).drop(columns=[target])
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, y_train, y_test

# Austin SVR regression data
X_train_austin_reg, X_test_austin_reg, y_train_austin_reg, y_test_austin_reg = preprocess_for_svr(
    austin_df,
    target='latestPrice',
    drop_cols=['city', 'latest_saledate', 'high_price']
)

# Airbnb SVR regression data
airbnb_regression_drop_cols = airbnb_df.select_dtypes(include=['object']).columns.tolist() + ['id', 'high_price']
X_train_airbnb_reg, X_test_airbnb_reg, y_train_airbnb_reg, y_test_airbnb_reg = preprocess_for_svr(
    airbnb_df,
    target='price',
    drop_cols=airbnb_regression_drop_cols
)


# ----------- SVR Regression Modeling -----------
svr_model = SVR(kernel='rbf', C=1.0, gamma='scale')

# Austin SVR regression
svr_model.fit(X_train_austin_reg, y_train_austin_reg)
y_pred_austin_reg = svr_model.predict(X_test_austin_reg)
print("\n----- Austin SVR Regression Results -----")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_austin_reg, y_pred_austin_reg)):.2f}")
print(f"MAE: {mean_absolute_error(y_test_austin_reg, y_pred_austin_reg):.2f}")
print(f"R2 Score: {r2_score(y_test_austin_reg, y_pred_austin_reg):.4f}")

# Airbnb SVR regression
svr_model.fit(X_train_airbnb_reg, y_train_airbnb_reg)
y_pred_airbnb_reg = svr_model.predict(X_test_airbnb_reg)
print("\n----- Airbnb SVR Regression Results -----")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_airbnb_reg, y_pred_airbnb_reg)):.2f}")
print(f"MAE: {mean_absolute_error(y_test_airbnb_reg, y_pred_airbnb_reg):.2f}")
print(f"R2 Score: {r2_score(y_test_airbnb_reg, y_pred_airbnb_reg):.4f}")


----- Austin SVR Regression Results -----
RMSE: 498133.91
MAE: 213610.73
R2 Score: -0.0484

----- Airbnb SVR Regression Results -----
RMSE: 488.67
MAE: 109.29
R2 Score: 0.0269
